In [1]:
# Maria Júlia Testoni, Lucas Queiroz, Felipe Carvalho
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://dippg.cefet-rj.br/ppdsp/index.php/pt/defesas"

# Usa o Session para melhorar o acesso e o processamento
session = requests.Session()
# acessa a primeira url
response = session.get(url, verify=False)

soup = BeautifulSoup(response.content, "html.parser")

# coleta apenas a tag a, visto que é uma lista e queremos somente os links
links = soup.find_all("a")
lista_links = []

for link in links:
  # coleta o link
    href = link.get("href")

    # focando apenas em defesa, o pegamos (assim ajudou a entrar somente nesse assunto e não ter problema de acesso por ser link inválido)
    if href and "defesa" in href.lower():
        lista_links.append(urljoin(url, href))

# remove links duplicados
lista_links = list(set(lista_links))

# limite de 15 (para processar melhor)
for link in lista_links[:15]:

    try:
        # acessa cada link com cada conteudo
        pagina = session.get(link, verify=False)
        soup_pagina = BeautifulSoup(pagina.content, "html.parser")

        # o titulo esta dentro de uma tag a, dentro de uma tag h1
        titulo_tag = soup_pagina.select_one("h1 a")

        if titulo_tag:
            titulo = titulo_tag.get_text(strip=True)
        else:
            titulo = "Sem título"

        # procura a tag p com a palavra Resumo, onde começa nosso resumo
        p_resumo = soup_pagina.find("p", string=lambda t: t and "Resumo" in t)

        resumo = ""

        if p_resumo:
            # pula o primeiro p, após a tag Resumo
            p1 = p_resumo.find_next("p")
            if p1:
                # chega ao p que possui o resumo (conteudo)
                p2 = p1.find_next("p")
                if p2:
                    resumo = p2.get_text(strip=True)

        # monta saida
        if resumo:
            print("LINK:", link)
            print("TÍTULO:", titulo)
            print("RESUMO:", resumo)
            print("-" * 80)

    except Exception as e:
        print("Erro ao acessar:", link)
        print(e)

LINK: https://dippg.cefet-rj.br/ppdsp/index.php/pt/defesas/182-defesa-de-dissertacao-de-leonardo-da-silva-araujo
TÍTULO: Defesa de Dissertação de Leonardo da Silva Araújo
RESUMO: O transporte e a saúde são direitos sociais garantidos pela Constituição Brasileira a todos os cidadãos e que muitas das vezes se entrelaçam. A mobilidade urbana, por exemplo, é peça chave para permitir o acesso das pessoas aos serviços básicos de saúde. Por esse motivo, o objetivo desse trabalho foi analisar questões relacionadas ao transporte ativo e a saúde na região da Baixada Fluminense (RJ) por meio de três artigos. No primeiro artigo foi feita uma análise bibliométrica que buscou os principais trabalhos relacionados a bicicleta, ciclismo e ambiente construído e encontrou-se como principais resultados trabalhos que relacionavam esses tópicos à saúde e atividade física. O artigo também evidenciou que as publicações sobre ciclismo e ambiente construído estão em crescimento, especialmente em países desenvol